<a href="https://colab.research.google.com/github/Julkinis/ab_testing_ecommerce/blob/main/z_test_conv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Загрузка данных

In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("faviovaz/marketing-ab-testing")

print("Path to dataset files:", path)

100%|██████████| 5.23M/5.23M [00:00<00:00, 148MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/faviovaz/marketing-ab-testing/versions/1


In [13]:
df = pd.read_csv('/root/.cache/kagglehub/datasets/faviovaz/marketing-ab-testing/versions/1/marketing_AB.csv')
df = df.drop(columns="Unnamed: 0")

**user id**: User ID (unique)

**test group:** If "ad" the person saw the advertisement, if "psa" they only saw the public service announcement

**converted:** If a person bought the product then True, else is False

**total ads:** Amount of ads seen by person

**most ads day:** Day that the person saw the biggest amount of ads

**most ads hour:** Hour of day that the person saw the biggest amount of ads

In [14]:
df.head()

,user id,test group,converted,total ads,most ads day,most ads hour
0,1069124,ad,False,130,Monday,20
1,1119715,ad,False,93,Tuesday,22
2,1144181,ad,False,21,Tuesday,18
3,1435133,ad,False,355,Tuesday,10
4,1015700,ad,False,276,Friday,14


# Анализ данных

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 588101 entries, 0 to 588100
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   user id        588101 non-null  int64 
 1   test group     588101 non-null  object
 2   converted      588101 non-null  bool  
 3   total ads      588101 non-null  int64 
 4   most ads day   588101 non-null  object
 5   most ads hour  588101 non-null  int64 
dtypes: bool(1), int64(3), object(2)
memory usage: 23.0+ MB


В данных нет пропусков, количество строк 588101


In [38]:
# Размер групп и баланс (проверка SRM)
n_ad, n_psa = df["test group"].value_counts()
n_ad, n_psa

(564577, 23524)

Группа psa в 24 раза меньше группы ad


In [43]:
# Конверсия по группам (использую весто дисперсии)
conv = df.groupby("test group")["converted"].agg(["count", "sum", "mean"])
conv_ad, conv_psa = conv.loc["ad", "mean"], conv.loc["psa", "mean"]
conv

,count,sum,mean
test group,,,
ad,564577,14423,0.025547
psa,23524,420,0.017854


In [49]:
import numpy as np
from scipy.stats import norm

# фиксируем уровень значимости альфа
alpha = 0.05
power = 0.8

z_alpha = norm.ppf(1 - alpha / 2)
z_beta = norm.ppf(power)

mde = (z_alpha + z_beta) * np.sqrt(
    conv_psa * (1 - conv_psa) * (1 / n_ad + 1 / n_psa)
)

mde

np.float64(0.002468707761457463)

0.24% изменений мы способны отловить

In [50]:
# Распределение total ads - общее количество рекламы просомтренный пользователем
df.groupby("test group")["total ads"].describe()

,count,mean,std,min,25%,50%,75%,max
test group,,,,,,,,
ad,564577.0,24.823365,43.750456,1.0,4.0,13.0,27.0,2065.0
psa,23524.0,24.761138,42.860720,1.0,4.0,12.0,26.0,907.0


# Эксперимент

H₀: конверсия в группе с рекламой равна конверсии в контрольной группе. Реклама ничего не меняет.

H₁: конверсии различаются. Тест двусторонний, реклама может как увеличить, так и уменьшить конверсию


In [52]:
ad = df[df["test group"] == "ad"]["converted"]
psa = df[df["test group"] == "psa"]["converted"]

count = [ad.sum(), psa.sum()]
nobs = [len(ad), len(psa)]

count, nobs

([np.int64(14423), np.int64(420)], [564577, 23524])

In [56]:
from statsmodels.stats.proportion import proportions_ztest

# для анализа конверсии выбран z-test
stat, p_value = proportions_ztest(count, nobs)

stat, p_value

(np.float64(7.3700781265454145), np.float64(1.7052807161559727e-13))

In [58]:
alpha - p_value

np.float64(0.04999999999982947)

Так как p_value << alpha, есть основания отвергнуть Н0

In [61]:
ci = confint_proportions_2indep(count[0], nobs[0], count[1], nobs[1], method="wald")
ci

(np.float64(0.005950932431611005), np.float64(0.00943397395279203))

Разница конверсий лежит примерно от 0,60 до 0,94 п.п.

Конверсия в группе с рекламой (ad) составила 2,55% против 1,79% в контрольной группе (psa).

Разница конверсий составила 0,77 п.п. (95% ДИ: 0,60–0,94 п.п.), что указывает на статистически значимый положительный эффект рекламы
